In [ ]:
import numpy as np
from PIL import Image
import cv2
from sklearn.neighbors import NearestNeighbors

In [ ]:
def set_corr_manual():
    """
    Specify correspondences between images

    Returns
    -------
    p1 : ndarray of shape (n, 2)
        Matched keypoint locations in image 1
    p2 : ndarray of shape (n, 2)
        Matched keypoint locations in image 2
    """
    
    input_img = Image.open("00003.png")

    w, h =input_img.size

    # p1이 base, p2가 input image
    # p1이 00002.png, p2가 00003.png

    p1 = np.array([[328, 79], [457, 82], [454, 200], [328, 198]], dtype=float)

    p2 = np.array([[0, 0], [w-1, 0], [w-1, h-1], [0, h-1]], dtype=float)
    
    return p1, p2

In [ ]:
def compute_H(p1, p2):
    """
    Estimate the homography between images

    Parameters
    ----------
    p1 : ndarray of shape (n, 2)
        Matched keypoint locations in image 1
    p2 : ndarray of shape (n, 2)
        Matched keypoint locations in image 2

    Returns
    -------
    H : ndarray of shape (3, 3)
        The estimated homography
    """
    
    # helper func
    def normal(points):

        centroid = points.mean(axis=0)

        points_centered = points - centroid

        scale = np.sqrt(2) / np.sqrt((points_centered ** 2).sum(axis=1).mean())

        T = np.array([[scale, 0, -scale * centroid[0]],
                      [0, scale, -scale * centroid[1]],
                      [0, 0, 1]])
        
        return T
    
    T1 = normal(p1)
    p1_h = np.hstack([p1, np.ones((p1.shape[0], 1))])
    p1_normal = (T1 @ p1_h.T).T

    T2 = normal(p2)
    p2_h = np.hstack([p2, np.ones((p2.shape[0], 1))])
    p2_normal = (T2 @ p2_h.T).T

    X = np.zeros((2 * len(p1), 9))

    for i in range(len(p1)):

        x1, y1 = p1_normal[i, 0], p1_normal[i, 1]
        x2, y2 = p2_normal[i, 0], p2_normal[i, 1]

        X[2*i] = [-x1, -y1, -1, 0, 0, 0, x2*x1, x2*y1, x2]
        X[2*i + 1] = [0, 0, 0, -x1, -y1, -1, y2*x1, y2*y1, y2]

    U, S, Vt = np.linalg.svd(X)
    H_noraml = Vt[-1].reshape(3, 3)
    H = np.linalg.inv(T2) @ H_noraml @ T1

    return H / H[2, 2]

In [ ]:
def insert_image(base_img, input_img, H):
    """
    Insert projected input_img in base_img with estimated homography

    Parameters
    ----------
    base_img : ndarray of shape (h, w, 3), base image
    input_img : ndarray of image to be projected and inserted
    H : ndarray of shape (3, 3)
        The estimated homography

    Returns
    -------
    merged_img : ndarray of shape (h, w, 3)
    """
    
    h_base, w_base = base_img.shape[:2]
    h_input, w_input = input_img.shape[:2]

    y, x = np.meshgrid(np.arange(h_base), np.arange(w_base), indexing='ij')

    coordinate = np.stack([x.ravel(), y.ravel(), np.ones(h_base * w_base)])
    
    source_coordinate = H @ coordinate
    source_coordinate /= source_coordinate[2]
    source_x = source_coordinate[0].reshape(h_base, w_base)
    source_y = source_coordinate[1].reshape(h_base, w_base)

    valid_pixel = (source_x >= 0) & (source_x < w_input) & (source_y >= 0) & (source_y < h_input)

    source_x = np.clip(np.round(source_x).astype(int), 0, w_input - 1)
    source_y = np.clip(np.round(source_y).astype(int), 0, h_input - 1)

    merged_img = base_img.copy()

    merged_img[valid_pixel] = input_img[source_y[valid_pixel], source_x[valid_pixel]]

    return merged_img

In [ ]:
# Q 2-1

# Read images
base_img = Image.open('00002.png').convert('RGB')
# Choose any image you want to insert and save it as 00003.png in this folder.
input_img = Image.open('00003.png').convert('RGB')
base_img = np.array(base_img)
input_img = np.array(input_img)

# Set correspondences manually
p1, p2 = set_corr_manual()

# Estimate the homography between images
H = compute_H(p1, p2)

# Insert warped input_img to base_img
merged_img = insert_image(base_img, input_img, H)
Image.fromarray(np.uint8(merged_img)).save('output_2-1.png')


In [ ]:
def match_sift(loc1, des1, loc2, des2, distance_ratio):
    """
    Find the matches of SIFT features between two images

    Parameters
    ----------
    loc1 : ndarray of shape (n1, 2)
        Keypoint locations in image 1
    des1 : ndarray of shape (n1, 128)
        SIFT descriptors of the keypoints image 1
    loc2 : ndarray of shape (n2, 2)
        Keypoint locations in image 2
    des2 : ndarray of shape (n2, 128)
        SIFT descriptors of the keypoints image 2
    distance_ratio : threshold for the ratio test

    Returns
    -------
    x1 : ndarray of shape (n, 2)
        Matched keypoint locations in image 1
    x2 : ndarray of shape (n, 2)
        Matched keypoint locations in image 2
    """
    
    NN1 = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(des2)
    distances, indices = NN1.kneighbors(des1)

    NN2 = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(des1)
    distances_reverse, indices_reverse = NN2.kneighbors(des2)

    x1_list = []
    x2_list = []

    for i in range(len(des1)):

        if distances[i, 0] / (distances[i, 1] + 1e-8) >= distance_ratio:
            continue

        j = indices[i, 0]

        if indices_reverse[j, 0] != i:
            continue

        x1_list.append(loc1[i])
        x2_list.append(loc2[j])

    return np.array(x1_list), np.array(x2_list)

In [ ]:
def compute_H_ransac(x1, x2, ransac_n_iter, ransac_thr):
    """
    Estimate the homography between images using RANSAC

    Parameters
    ----------
    x1 : ndarray of shape (n, 2)
        Matched keypoint locations in image 1
    x2 : ndarray of shape (n, 2)
        Matched keypoint locations in image 2
    ransac_n_iter : int
        Number of RANSAC iterations
    ransac_thr : float
        Error threshold for RANSAC

    Returns
    -------
    H : ndarray of shape (3, 3)
        The estimated homography
    """

    n = len(x1)

    best_inliers = []
    best_count = 0

    x1_h = np.hstack([x1, np.ones((n, 1))])

    for _ in range(ransac_n_iter):

        indices = np.random.choice(n, 4, replace=False)

        H = compute_H(x1[indices], x2[indices])

        x2_proj = (H @ x1_h.T).T

        x2_proj /= x2_proj[:, 2:3]
        
        error = np.sqrt(((x2_proj[:, :2] - x2) ** 2).sum(axis=1))

        inliers = error < ransac_thr
        count = inliers.sum()

        if count > best_count:

            best_count = count
            best_inliers = inliers
    
    return compute_H(x1[best_inliers], x2[best_inliers])

In [ ]:
def merge_image(img_1, img_2, H):
    """
    Merge projected input_img with base_img using estimated homography.

    Parameters
    ----------
    img_1 : ndarray of shape (h, w, 3), base image
    img_2 : ndarray of shape (h, w, 3), image to be projected and merged with base image
    H : ndarray of shape (3, 3)
        The estimated homography

    Returns
    -------
    merged_img : ndarray of shape (2h, 2w, 3), base image placed at the top-left corner
    """
    
    h1, w1 = img_1.shape[:2]
    h2, w2 = img_2.shape[:2]

    merged_img = np.zeros((h1*2, w1*2, 3), dtype=np.uint8)

    y, x = np.meshgrid(np.arange(h1*2), np.arange(w1*2), indexing='ij')
    
    coordinate = np.stack([x.ravel(), y.ravel(), np.ones(h1 * w1 * 4)])

    source_coordinate = H @ coordinate
    source_coordinate /= source_coordinate[2]
    source_x = source_coordinate[0].reshape(h1*2, w1*2)
    source_y = source_coordinate[1].reshape(h1*2, w1*2)

    valid_pixel = (source_x >= 0) & (source_x < w2) & (source_y >= 0) & (source_y < h2)

    source_x = np.clip(np.round(source_x).astype(int), 0, w2 - 1)
    source_y = np.clip(np.round(source_y).astype(int), 0, h2 - 1)

    merged_img[valid_pixel] = img_2[source_y[valid_pixel], source_x[valid_pixel]]

    merged_img[:h1, :w1] = img_1
    return merged_img

In [ ]:
# Q 2-2

# Hyperparameters, feel free to modify
ransac_n_iter = 500
ransac_thr = 3
distance_ratio = 0.75

# Read images
img_1 = Image.open('00004.jpg').convert('RGB')
img_2 = Image.open('00005.jpg').convert('RGB')
img_1 = np.array(img_1)
img_2 = np.array(img_2)

# Extract SIFT features
sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(img_1, None)
kp2, des2 = sift.detectAndCompute(img_2, None)
loc1 = np.array([kp.pt for kp in kp1])
loc2 = np.array([kp.pt for kp in kp2])

# Find the matches between two images (x1 <--> x2)
x1, x2 = match_sift(loc1, des1, loc2, des2, distance_ratio)

# Estimate the homography between images using RANSAC
H = compute_H_ransac(x1, x2, ransac_n_iter, ransac_thr)

# Warp img_2 and merge with img_1
merged_img = merge_image(img_1, img_2, H) 
Image.fromarray(np.uint8(merged_img)).save('output_2-2.png')
